In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma
import math
import time

# =====================================================================
# CÉLULA 1: PREPARAÇÃO DOS DADOS E MOTOR TSK COM REGULARIZAÇÃO RIDGE
# =====================================================================
X_train = np.arange(1, 11, dtype=float)
y_factorial = np.array([math.factorial(int(n)) for n in X_train], dtype=float)

# Transformação de escala suavizada para evitar overflow numérico
y_train_transformed = np.zeros_like(X_train)
for i, n in enumerate(X_train):
    fact = math.factorial(int(n))
    y_train_transformed[i] = 1.0 if fact == 1 else 1.0 / np.log(fact)

def tsk_inference(X, centers, sigmas, y_target=None, ridge_lambda=1e-2):
    X = np.atleast_1d(X)
    N = len(X)
    R = len(centers)
    
    W = np.zeros((N, R))
    for i in range(N):
        for j in range(R):
            W[i, j] = np.exp(-((X[i] - centers[j])**2) / (2 * (sigmas[j]**2) + 1e-5))
            
    row_sums = W.sum(axis=1, keepdims=True)
    W_norm = np.where(row_sums > 1e-12, W / row_sums, 1.0 / R)
    
    X_hat = np.zeros((N, 2 * R))
    for j in range(R):
        X_hat[:, 2*j] = W_norm[:, j] * X
        X_hat[:, 2*j + 1] = W_norm[:, j]
        
    if y_target is not None:
        A = X_hat.T @ X_hat + ridge_lambda * np.eye(2 * R)
        try:
            P = np.linalg.inv(A) @ X_hat.T @ y_target
        except np.linalg.LinAlgError:
            P = np.linalg.pinv(A) @ X_hat.T @ y_target
        return X_hat @ P, P
    else:
        return X_hat

# =====================================================================
# CÉLULA 2: IMPLEMENTAÇÃO DO METODO BIOINSPIRADO 1 - PSO
# =====================================================================
def particle_swarm_optimization(X, y_trans, num_rules, pop_size=40, iterations=40, seed=42):
    np.random.seed(seed)
    centers = np.linspace(X.min(), X.max(), num_rules)
    
    w, c1, c2 = 0.6, 1.8, 1.8
    position = np.random.uniform(1.5, 4.0, size=(pop_size, num_rules))
    velocity = np.random.uniform(-0.2, 0.2, size=(pop_size, num_rules))
    
    pbest_position = position.copy()
    pbest_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in position])
    
    gbest_idx = np.argmin(pbest_fitness)
    gbest_position = pbest_position[gbest_idx].copy()
    
    history = []
    for it in range(iterations):
        for i in range(pop_size):
            r1, r2 = np.random.rand(num_rules), np.random.rand(num_rules)
            velocity[i] = w * velocity[i] + c1 * r1 * (pbest_position[i] - position[i]) + c2 * r2 * (gbest_position - position[i])
            position[i] = np.clip(position[i] + velocity[i], 1.0, 6.0)
            
            y_pred, _ = tsk_inference(X, centers, position[i], y_trans)
            current_mse = np.mean((y_trans - y_pred)**2)
            
            if current_mse < pbest_fitness[i]:
                pbest_fitness[i] = current_mse
                pbest_position[i] = position[i].copy()
                if current_mse < pbest_fitness[gbest_idx]:
                    gbest_position = position[i].copy()
                    gbest_idx = i
        history.append(pbest_fitness[gbest_idx])
    return gbest_position, history, centers

# =====================================================================
# CÉLULA 3: IMPLEMENTAÇÃO DO MÉTODO EVOLUTIVO 2 - ALGORITMO GENÉTICO (GA)
# =====================================================================
def genetic_algorithm(X, y_trans, num_rules, pop_size=40, generations=40, mutation_rate=0.2, seed=42):
    np.random.seed(seed)
    centers = np.linspace(X.min(), X.max(), num_rules)
    
    # Inicialização uniforme idêntica ao espaço do PSO
    population = np.random.uniform(1.5, 4.0, size=(pop_size, num_rules))
    history = []
    
    for gen in range(generations):
        fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in population])
        best_idx = np.argmin(fitness)
        history.append(fitness[best_idx])
        
        # Seleção por Torneio
        new_population = []
        for _ in range(pop_size):
            candidates = np.random.choice(pop_size, size=3, replace=False)
            winner = candidates[np.argmin(fitness[candidates])]
            new_population.append(population[winner].copy())
            
        # Crossover Aritmético BLX-alpha simplificado
        new_population = np.array(new_population)
        for i in range(0, pop_size, 2):
            if i+1 < pop_size and np.random.rand() < 0.8:
                alpha = np.random.rand()
                child1 = alpha * new_population[i] + (1 - alpha) * new_population[i+1]
                child2 = alpha * new_population[i+1] + (1 - alpha) * new_population[i]
                new_population[i] = np.clip(child1, 1.0, 6.0)
                new_population[i+1] = np.clip(child2, 1.0, 6.0)
                
        # Mutação Gaussiana Controlada
        for i in range(pop_size):
            if np.random.rand() < mutation_rate:
                mutation_vector = np.random.normal(0, 0.2, size=num_rules)
                new_population[i] = np.clip(new_population[i] + mutation_vector, 1.0, 6.0)
                
        # Elitismo estrito
        new_population[0] = population[best_idx]
        population = new_population
        
    final_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in population])
    return population[np.argmin(final_fitness)], history, centers

In [ ]:
seeds = [10, 42, 100, 2026, 999]
NUM_REGRAS = 4
ITERACOES = 40

pso_mses, pso_times, pso_curves = [], [], []
ga_mses, ga_times, ga_curves = [], [], []

print("=== EXECUTANDO O ENXAME DE PARTÍCULAS (PSO) ===")
for s in seeds:
    t0 = time.time()
    _, hist, _ = particle_swarm_optimization(X_train, y_train_transformed, num_rules=NUM_REGRAS, iterations=ITERACOES, seed=s)
    pso_times.append(time.time() - t0)
    pso_mses.append(hist[-1])
    pso_curves.append(hist)
print("PSO Concluído.")

print("\n=== EXECUTANDO O ALGORITMO GENÉTICO (GA) ===")
for s in seeds:
    t0 = time.time()
    _, hist, _ = genetic_algorithm(X_train, y_train_transformed, num_rules=NUM_REGRAS, generations=ITERACOES, seed=s)
    ga_times.append(time.time() - t0)
    ga_mses.append(hist[-1])
    ga_curves.append(hist)
print("GA Concluído.")

# Exibição Estatística Comparativa para preencher as tabelas do Overleaf
print("\n" + "="*50)
print("        TABELA COMPARATIVA ESTATÍSTICA")
print("="*50)
print(f"Métrica               | PSO              | GA")
print(f"--------------------------------------------------")
print(f"Melhor MSE            | {np.min(pso_mses):.6f}         | {np.min(ga_mses):.6f}")
print(f"Pior MSE              | {np.max(pso_mses):.6f}         | {np.max(ga_mses):.6f}")
print(f"Média MSE             | {np.mean(pso_mses):.6f}         | {np.mean(ga_mses):.6f}")
print(f"Desvio Padrão MSE     | {np.std(pso_mses):.6f}         | {np.std(ga_mses):.6f}")
print(f"Tempo Médio (s)       | {np.mean(pso_times):.4f}           | {np.mean(ga_times):.4f}")
print("="*50)

In [ ]:
# Recuperação dos melhores modelos de cada categoria
best_pso_idx = np.argmin(pso_mses)
best_pso_sigmas, _, pso_centers = particle_swarm_optimization(X_train, y_train_transformed, num_rules=NUM_REGRAS, iterations=ITERACOES, seed=seeds[best_pso_idx])
_, optimal_P_pso = tsk_inference(X_train, pso_centers, best_pso_sigmas, y_train_transformed)

best_ga_idx = np.argmin(ga_mses)
best_ga_sigmas, _, ga_centers = genetic_algorithm(X_train, y_train_transformed, num_rules=NUM_REGRAS, generations=ITERACOES, seed=seeds[best_ga_idx])
_, optimal_P_ga = tsk_inference(X_train, ga_centers, best_ga_sigmas, y_train_transformed)

# Geração do domínio contínuo para plotagem
X_continuous = np.linspace(1.1, 10.0, 300)

# Inferências Contínuas
y_pso_trans = tsk_inference(X_continuous, pso_centers, best_pso_sigmas) @ optimal_P_pso
y_ga_trans = tsk_inference(X_continuous, ga_centers, best_ga_sigmas) @ optimal_P_ga

# Inversão Exponencial
y_pso_factorial = np.exp(1.0 / y_pso_trans)
y_ga_factorial = np.exp(1.0 / y_ga_trans)

# Renderização Gráfica dos Resultados
plt.figure(figsize=(18, 5))

# 1. Confronto de Convergência (Média das 5 execuções)
plt.subplot(1, 3, 1)
plt.plot(np.mean(pso_curves, axis=0), 'r-', linewidth=2, label='PSO (Média)')
plt.plot(np.mean(ga_curves, axis=0), 'b-', linewidth=2, label='GA (Média)')
plt.title("Velocidade de Convergência: PSO vs GA")
plt.xlabel("Iterações / Gerações")
plt.ylabel("MSE Médio (Espaço Transformado)")
plt.yscale('log')
plt.grid(True)
plt.legend()

# 2. Superfície de Controle no Espaço Intermediário
plt.subplot(1, 3, 2)
y_gamma_trans = 1.0 / np.log(gamma(X_continuous + 1))
plt.plot(X_continuous, y_gamma_trans, 'g-', label='Meta Analítica $1/\\ln(x!)$', alpha=0.5)
plt.plot(X_continuous, y_pso_trans, 'r--', label='Inferência PSO', linewidth=1.5)
plt.plot(X_continuous, y_ga_trans, 'b:', label='Inferência GA', linewidth=2)
plt.scatter(X_train, y_train_transformed, color='black', zorder=5, label='Pontos de Treino')
plt.title("Aproximação no Espaço Transformado")
plt.xlabel("Entrada ($x$)")
plt.ylabel("Valor")
plt.grid(True)
plt.legend()

# 3. Resultado Final na Escala Real do Fatorial
plt.subplot(1, 3, 3)
plt.plot(X_continuous, gamma(X_continuous + 1), 'g-', label='Função Gamma $\\Gamma(x+1)$', alpha=0.6)
plt.plot(X_continuous, y_pso_factorial, 'r--', label='Reconstrução PSO', linewidth=1.5)
plt.plot(X_continuous, y_ga_factorial, 'b:', label='Reconstrução GA', linewidth=2)
plt.scatter(X_train, y_factorial, color='black', zorder=5, label='Fatoriais Reais ($n!$)')
plt.title("Aproximação Final Reconvertida")
plt.xlabel("Entrada ($x$)")
plt.ylabel("Valor (Escala Logarítmica)")
plt.yscale('log')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()